# 01 - Data understanding: NUCLE

Use the shared parser and CSV schema for notebook exploration and the report. Titles occupy text block 0 when present. Internal whitespace is preserved. Run cells from top to bottom. Requires pandas.

If a cell fails, fix the error, restart the kernel, and run all cells from the top. Paths come from the shared parser and currently use `data_set/`.


In [ ]:
from pathlib import Path
from collections import Counter
import sys

# Jupyter has no __file__; locate the repository from the kernel directory.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/data/parse_nucle.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook with the kernel working directory inside the project."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data.nucle_sgml import load_nucle
from src.data.parse_nucle import (
    documents_to_dataframe, TARGET_LABELS, DEFAULT_INPUT, DEFAULT_OUTPUT,
)
from src.data.data_understanding import build_report

DATA_PATH = DEFAULT_INPUT
OUTPUT_PATH = DEFAULT_OUTPUT
docs = load_nucle(DATA_PATH)
if not docs:
    raise ValueError("No documents found")
df = documents_to_dataframe(docs)
print(f"Documents: {len(docs):,}; annotations: {len(df):,}")
print("First text block:", docs[0].paragraphs[0] if docs[0].paragraphs else "<missing>")
print(df[["essay_id", "error_span", "error_type", "correction", "valid_offset"]].head(3).to_string(index=False))


## Error distribution and examples

Each annotation is a separate observation; annotations can share text. Target counts include rows flagged for review.


In [ ]:
print(df.error_type.value_counts().to_string())
target_df = df[df.is_target_label]
coverage = len(target_df) / len(df) if len(df) else 0
print(f"Target coverage: {len(target_df):,}/{len(df):,} ({coverage:.2%})")
for label in sorted(TARGET_LABELS):
    print("\n" + label)
    examples = target_df[(target_df.error_type == label) & target_df.valid_offset & ~target_df.cross_paragraph]
    print(examples[["error_span", "correction"]].head(3).to_string(index=False))


## Percentage distribution of the five selected labels

Each row counts annotations for one selected label. `percent_of_all` uses all corpus annotations as the denominator; `percent_of_selected` uses only annotations belonging to the five selected labels. Counts include annotations flagged for review. Percentages are displayed to two decimal places; rounded values may not sum to exactly 100%.


In [ ]:
selected_labels = list(TARGET_LABELS)
selected_counts = (
    df.loc[df["error_type"].isin(selected_labels), "error_type"]
    .value_counts()
    .reindex(selected_labels, fill_value=0)
)
label_percentages = selected_counts.rename_axis("error_type").reset_index(name="count")
total_annotations = len(df)
selected_annotations = int(selected_counts.sum())
label_percentages["percent_of_all"] = (
    label_percentages["count"] / total_annotations * 100
    if total_annotations else 0.0
)
label_percentages["percent_of_selected"] = (
    label_percentages["count"] / selected_annotations * 100
    if selected_annotations else 0.0
)
print(label_percentages.to_string(index=False, formatters={
    "percent_of_all": "{:.2f}%".format,
    "percent_of_selected": "{:.2f}%".format,
}))
selected_coverage = selected_annotations / total_annotations * 100 if total_annotations else 0.0
print(f"\nSelected labels: {selected_annotations:,}/{total_annotations:,} annotations ({selected_coverage:.2f}%).")


## Shared data understanding report

This is the same report generator used by `python -m src.data.data_understanding`. Span edit categories do not necessarily identify linguistic missing-word errors.


In [ ]:
print(build_report(df))


## Optional export

Export uses the shared schema expected by the report script. Split by essay ID during later preparation and fit learned preprocessing on training data only.


In [ ]:
EXPORT_CSV = False
if EXPORT_CSV:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {len(df):,} annotations to {OUTPUT_PATH}")
else:
    print("Export disabled. Set EXPORT_CSV = True to save the shared annotation table.")
